# Radio Life Writing DE: Selection and Semantic Map Method

This notebook explains and reproduces, as far as possible from the checked-in data, how the public semantic map for **Radio Life Writing DE** was made.

The aim is not to present the map as a finished canon. The map is a curated research interface: it starts from a broad scrape of German radio metadata, filters likely life-writing records, models the high- and medium-confidence subset, and exports a manually reviewed public map.

## What this notebook covers

1. Where the records came from.
2. How the first biographical selection was made.
3. How the high- and medium-confidence records became the modeling core.
4. How the public 566-record map differs from the larger internal candidate set.
5. How clusters and manual curation should be read.

The notebook uses the repository data files when they are available locally. If a file is missing, the explanatory cells still describe the method.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

# Adjust this if you move the notebook somewhere else.
CANDIDATE_ROOTS = [
    Path(".."),
    Path("."),
    Path("../work/radio-life-writing-de-main"),
    Path("work/radio-life-writing-de-main"),
    Path("radio-life-writing-de-main"),
]

PROJECT_ROOT = next((path.resolve() for path in CANDIDATE_ROOTS if path.exists()), None)
PROJECT_ROOT

In [ ]:
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the repository data folder. Run this notebook from "
        "the repository root/notebooks folder, or update CANDIDATE_ROOTS."
    )

def read_json(relative_path):
    path = PROJECT_ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

def read_csv(relative_path):
    path = PROJECT_ROOT / relative_path
    return pd.read_csv(path) if path.exists() else None

scrape_summary = read_json("data/processed/german_scrape_summary.json")
deep_summary = read_json("data/processed/analysis_tables/deep_analysis_summary.json")
semantic_summary = read_json("data/processed/analysis_tables/semantic_model_summary.json")

candidates = read_csv("data/processed/german_radio_biographical_candidates.csv")
core = read_csv("data/processed/analysis_tables/core_semantic_enriched.csv")
visible_map = read_csv("data/latent_semantic_map_all_602_sorted.csv")
exclusions = read_csv("data/curation/life_writing_exclusions.csv")
subject_overrides = read_csv("data/curation/life_subject_overrides.csv")

scrape_summary

## Selection in one sentence

The public semantic map is a curated subset: **1,764 scraped German records** were reduced to **806 high- and medium-confidence modeling records**, then to **566 manually reviewed public map records**.

In [ ]:
summary_table = pd.DataFrame(
    [
        {"stage": "Merged German scrape", "records": scrape_summary.get("rows") if scrape_summary else len(candidates)},
        {"stage": "High/medium modeling core", "records": deep_summary.get("core_rows") if deep_summary else len(core)},
        {"stage": "Curated public semantic map", "records": len(visible_map)},
    ]
)
summary_table

## 1. Source collection

The corpus is German-first and draws on three source layers:

- **DRA / ARD Hoerspieldatenbank**: the historical catalogue backbone.
- **Hoerspiel und Feature / DLF Kultur**: contemporary programme and archive pages.
- **Wirklichkeit im Radio**: a smaller but text-rich source for documentary and feature work.

The pipeline used metadata and page text: titles, descriptions, tags, dates, durations, genres, authors, broadcasters, and related fields. It did **not** download audio files.

In [ ]:
source_counts = candidates["source_archive"].value_counts().rename_axis("source").reset_index(name="records")
source_counts

### How the sources were found

**DRA / ARD Hoerspieldatenbank** records were found through search-form queries for life-writing terms such as biography, autobiography, portrait, memory, diary, letters, and related title terms. The scraper followed detail-page links and parsed the catalogue metadata.

**Hoerspiel und Feature / DLF Kultur** records came from programme sitemaps plus search results for biography-oriented terms.

**Wirklichkeit im Radio** records came from the site's WordPress sitemap for programme pages.

## 2. First-pass biographical scoring

The first selection gate is deliberately simple. It uses keyword-pattern scoring over searchable metadata and programme text.

A record is classified as:

- **high-confidence biographical** when two or more life-writing patterns are found;
- **possible biographical** when one pattern is found;
- **not biographical** when no pattern is found.

This is a triage method. It is useful for making a large archive browsable, but it is not a final scholarly judgment about every record.

In [ ]:
GERMAN_PATTERNS = [
    r"\bleben\b",
    r"\bbiograph(?:ie|isch)\b",
    r"\bautobiograph(?:ie|isch)\b",
    r"\bportr[aä]t\b",
    r"\btagebuch\b",
    r"\bbriefe\b",
    r"\berinnerungen\b",
]

ENGLISH_PATTERNS = [
    r"\blife of\b",
    r"\blives of\b",
    r"\bbiograph(?:y|ical)\b",
    r"\bautobiograph(?:y|ical)\b",
    r"\bportrait(?: of)?\b",
    r"\bmemoir\b",
    r"\bdiar(?:y|ies)\b",
    r"\bletters(?: of)?\b",
    r"\bbased on the life\b",
]

patterns = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ENGLISH_PATTERNS + GERMAN_PATTERNS]

def biographical_score(text):
    return sum(1 for pattern in patterns if pattern.search(str(text or "")))

def biographical_label(text):
    score = biographical_score(text)
    if score >= 2:
        return "high_confidence_biographical"
    if score == 1:
        return "possible_biographical"
    return "not_biographical"

pd.DataFrame({"German patterns": GERMAN_PATTERNS})

In [ ]:
label_counts = candidates["biographical_label"].value_counts().rename_axis("label").reset_index(name="records")
confidence_counts = candidates["candidate_confidence"].value_counts().rename_axis("confidence").reset_index(name="records")

display(label_counts)
display(confidence_counts)

## 3. The modeling core

Only **high** and **medium** confidence records enter the modeling core. Low-confidence and non-life-writing records are held out of the public semantic map.

This is why the model is best read as a browsing and discovery instrument: it helps locate patterns among plausible life-writing records, rather than proving that all scraped records are biographical.

In [ ]:
core_check = candidates[candidates["candidate_confidence"].isin(["high", "medium"])]
pd.DataFrame(
    {
        "measure": ["records in candidate file marked high/medium", "records in modeled core table"],
        "records": [len(core_check), len(core)],
    }
)

In [ ]:
core_by_source = core["source_archive"].value_counts().rename_axis("source").reset_index(name="core records")
core_by_source

## 4. Text modeling pipeline

The semantic model uses a standard exploratory text-analysis workflow:

1. Clean boilerplate from metadata text.
2. Convert text to TF-IDF features.
3. Reduce dimensionality with truncated SVD.
4. Group records into working clusters.
5. Project records into 2D with t-SNE.
6. Use broadcast year as the vertical axis in the public 3D map.

The public map should therefore be read as an interpretive navigation layer, not as a measurement instrument. Nearby points are useful prompts for comparison, but t-SNE distances should not be overinterpreted.

In [ ]:
semantic_summary

## 5. Public map clusters

For the public article, the visible map uses curated cluster labels for readability. In this notebook, we focus on the labels that appear in the public 566-record map, not on internal modeling artifacts.

In [ ]:
visible_clusters = (
    visible_map["cluster"]
    .value_counts()
    .rename_axis("public cluster")
    .reset_index(name="visible records")
)
visible_clusters

The labels are working titles. They were assigned by inspecting top terms, examples, source mix, and manual curation notes.

Examples of interpretive rules:

- terms such as `briefe`, `brief`, or `briefwechsel` support **Letters and correspondence**;
- terms such as `tagebuch` or `tagebücher` support **Diaries and self-records**;
- terms such as `porträt`, `portrait`, or `portrat` support **Portrait catalogue**;
- terms such as `autobiographie`, `biographie`, `biografie`, or `leben` support **Auto/biographical lives**.

## 6. Manual curation layer

After computational triage, the project uses manual curation files. These correct, exclude, or enrich records before they appear in the public map.

The curation layer includes:

- exclusions;
- subject/protagonist overrides;
- year overrides;
- source-card descriptions;
- protagonist audits.

In [ ]:
curation_counts = pd.DataFrame(
    [
        {"curation file": "life_writing_exclusions.csv", "rows": len(exclusions) if exclusions is not None else None},
        {"curation file": "life_subject_overrides.csv", "rows": len(subject_overrides) if subject_overrides is not None else None},
    ]
)
curation_counts

In [ ]:
if exclusions is not None:
    display(exclusions.head(10))

if subject_overrides is not None:
    display(subject_overrides.head(10))

## 7. Final public export

The public map is exported from `data/latent_semantic_map_all_602_sorted.csv`.

That spreadsheet supplies the curated public-facing metadata: title, year, source, genre, protagonist, role, author, director, URL, and description. The exporter then restores the x/y map position by matching each URL to the earlier modeled core table, maps broadcast year to the z-axis, and writes the browser file `article/semantic-map-3d-data.js`.

In [ ]:
visible_map[[
    "map_order",
    "title",
    "year",
    "source",
    "cluster",
    "genre",
    "protagonist_verified",
    "protagonist_role",
    "url",
]].head(12)

## Suggested wording for the method section

The final semantic map shows 566 curated records, but these records come from a larger pipeline. First, 1,764 German radio records were collected from DRA / ARD Hoerspieldatenbank, Hoerspiel und Feature / DLF Kultur, and Wirklichkeit im Radio. The pipeline worked from archive and programme metadata, descriptions, page text, tags, dates, durations, genres, authors, and similar textual fields; no audio files were downloaded.

The main selection criterion was confidence. High-confidence records contain explicit biographical signals in source metadata or programme text. Medium-confidence records contain weaker but plausible life-writing evidence. Low-confidence and non-life-writing items were excluded from the public semantic map.

The high- and medium-confidence records were modeled through an exploratory text pipeline using TF-IDF, truncated SVD, working clusters, and t-SNE. The public map uses curated cluster labels as interpretive aids rather than fixed categories. These labels were checked against terms, examples, source composition, and manual curation notes.